In [1]:
import numpy as np
import pandas as pd
import xarray as xr
from itertools import product

In [3]:
# Load probability table (trained on data <= 2024)
final_path='../data/processed/SPY_cond_ex_tab.nc'
ds = xr.open_dataset(final_path)

# Load full price data
df = pd.read_csv(
    '../data/raw/SPY_daily_yahoo_raw.csv',
    skiprows=[0,1,2],
    header=None,
    names=['Date','Adj Close','Close','High','Low','Open','Volume'],
    index_col=0,
    parse_dates=True
)

df = df.sort_index()
prices = df['Adj Close']

In [4]:
prices_2025 = prices.loc['2025-01-01':'2025-12-31']

In [5]:
directions = ["+", "-"]
q_vals = np.arange(0.1, 1.0, 0.1)
r_vals = [0.8, 0.9]
taus = [50, 100, 200, 250]
p_mins = [0.8, 0.9]
se_maxes = [0.01, 0.05, 0.1]

In [6]:
quantile_thresholds = {}

P = prices.values

for tau in taus:
    inc = P[tau:] - P[:-tau]
    for q in q_vals:
        quantile_thresholds[(tau, q, "+")] = np.quantile(inc, q)
        quantile_thresholds[(tau, q, "-")] = np.quantile(inc, 1 - q)

In [7]:
results = []

price_array = prices.values
date_index = prices.index

for direction, q, r, tau, p_min, se_max in product(
    directions, q_vals, r_vals, taus, p_mins, se_maxes
):

    # --- lookup probability & uncertainty ---
    p_hat = ds.prob.sel(direction=direction, tau=tau, q=q, r=r).item()
    p_se = ds.prob_se.sel(direction=direction, tau=tau, q=q, r=r).item()

    # --- strategy eligibility ---
    if np.isnan(p_hat) or p_hat < p_min or p_se > se_max:
        results.append({
            "direction": direction,
            "q": q,
            "r": r,
            "tau": tau,
            "p_min": p_min,
            "se_max": se_max,
            "trades": 0,
            "total_return": 0.0,
            "sharpe": np.nan
        })
        continue

    # --- trading ---
    returns = []
    i = 0

    while i < len(prices_2025):

        date = prices_2025.index[i]
        idx = date_index.get_loc(date)

        if idx - tau < 0 or idx + tau >= len(price_array):
            i += 1
            continue

        inc_now = price_array[idx] - price_array[idx - tau]
        thresh = quantile_thresholds[(tau, q, direction)]

        cond = (
            inc_now >= thresh if direction == "+" else inc_now <= thresh
        )

        if not cond:
            i += 1
            continue

        # --- enter trade ---
        entry_price = price_array[idx]
        exit_price = price_array[idx + tau]

        if direction == "+":
            ret = (exit_price - entry_price) / entry_price
        else:
            ret = (entry_price - exit_price) / entry_price

        returns.append(ret)

        # enforce non-overlapping trades
        i += tau

    # --- performance stats ---
    if len(returns) == 0:
        total_return = 0.0
        sharpe = np.nan
    else:
        returns = np.array(returns)
        total_return = np.prod(1 + returns) - 1

        if len(returns) > 1 and returns.std(ddof=1) > 0:
            sharpe = returns.mean() / returns.std(ddof=1)
        else:
            sharpe = np.nan

    results.append({
        "direction": direction,
        "q": q,
        "r": r,
        "tau": tau,
        "p_min": p_min,
        "se_max": se_max,
        "trades": len(returns),
        "total_return": total_return,
        "sharpe": sharpe
    })

KeyError: "not all values found in index 'q'. Try setting the `method` keyword argument (example: method='nearest')."

In [ ]:
results_df = pd.DataFrame(results)

In [ ]:
results_df.sort_values("total_return", ascending=False).head(10)

In [ ]:
results_df.sort_values("sharpe", ascending=False).head(10)

In [ ]:
ds.close()